In [2]:
# This notebook will be used to run the regression models to look at annual growth
import sqlalchemy 
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from scipy import stats
import scikit_posthocs as sp
# import statsmodels.api as sm
import statsmodels.formula.api as smf # this is to run a lm like R

# company_df.to_sql(name = 'companies', con = engine, if_exists = 'append', index = False)


In [18]:
load_dotenv("../../.env")

mysql_host = os.environ.get("MYSQL_HOST")
mysql_user = os.environ.get("MYSQL_USER")
mysql_password = os.environ.get("MYSQL_PASSWORD")
mysql_database = os.environ.get("MYSQL_DATABASE")


#the f goes in front to embed variables
engine = sqlalchemy.create_engine(f"mysql+mysqlconnector://{mysql_user}:{mysql_password}@{mysql_host}/{mysql_database}")

with engine.connect() as conn:
    print("Connection successful")



Connection successful


In [19]:
#I am importing the data from MySQL and creating the daily returns variable

daily_prices = pd.read_sql("SELECT ticker, date, adjusted_close FROM daily_prices ", engine)

daily_prices.head()

daily_prices['time'] = daily_prices.groupby('ticker').cumcount() + 1
daily_prices['time_C'] = daily_prices['time'] - daily_prices['time'].mean() 
daily_prices.head(10)

,ticker,date,adjusted_close,time,time_C
0,DLR,2025-06-11,170.86,1,-125.5
1,EQIX,2025-06-11,873.83,1,-125.5
2,IRM,2025-06-11,97.56,1,-125.5
3,DLR,2025-06-12,171.53,2,-124.5
4,EQIX,2025-06-12,876.49,2,-124.5
5,IRM,2025-06-12,99.04,2,-124.5
6,DLR,2025-06-13,170.71,3,-123.5
7,EQIX,2025-06-13,872.79,3,-123.5
8,IRM,2025-06-13,98.15,3,-123.5
9,DLR,2025-06-16,172.26,4,-122.5


In [20]:
daily_prices.groupby('ticker')['adjusted_close'].mean()

ticker
DLR     171.759603
EQIX    858.760635
IRM     100.299365
Name: adjusted_close, dtype: float64

In [23]:

results_list = []


for ticker in daily_prices['ticker'].unique():
    temp_df = daily_prices[daily_prices['ticker'] == ticker]
    returnbytime = smf.ols("adjusted_close ~ time_C", data=temp_df)
    results = returnbytime.fit()
    results_list.append({
        'ticker': ticker,
        'regression_slope' : results.params['time_C'],
        'r_squared':  results.rsquared,
       # 'p_value_slope': results.pvalues['time'],
        #'p_value_f': results.f_pvalue,
        'intercept': results.params['Intercept']
            })
    print(f'\n\n the ticker is {ticker}', results.summary())
    
results_df = pd.DataFrame(results_list)
results_df.head()

with engine.begin() as conn:
    conn.execute(sqlalchemy.text("TRUNCATE TABLE regression_results"))
results_df.to_sql(name = 'regression_results', con = engine, if_exists = 'append', index = False)




 the ticker is DLR                             OLS Regression Results                            
Dep. Variable:         adjusted_close   R-squared:                       0.306
Model:                            OLS   Adj. R-squared:                  0.304
Method:                 Least Squares   F-statistic:                     110.4
Date:                Thu, 11 Jun 2026   Prob (F-statistic):           1.26e-21
Time:                        13:01:03   Log-Likelihood:                -951.88
No. Observations:                 252   AIC:                             1908.
Df Residuals:                     250   BIC:                             1915.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    171.7596      0.66

3

In [98]:
#Running the Kruskal-wallis test


# # Stability (Daily Range)
# #need the daily ranges (close-open)

# cv_data = pd.read_sql("SELECT ticker, cv_daily_range FROM monthly_summary", engine)

# # These are not statistically different from each other. Must interpret tomorrow
# # stats.kruskal(cv_data[cv_data['ticker'] == 'DLR']['cv_daily_range'], 
# #               cv_data[cv_data['ticker'] == 'EQIX']['cv_daily_range'], 
# #               cv_data[cv_data['ticker'] == 'IRM']['cv_daily_range'])

# # sp.posthoc_dunn(cv_data, 'cv_daily_range', 'ticker', p_adjust='bonferroni')

# # cv_data.groupby('ticker')['cv_daily_range'].median()
# stats.kruskal(daily_prices[daily_prices['ticker'] == 'DLR']['daily_returns'].dropna(), 
#               daily_prices[daily_prices['ticker'] == 'EQIX']['daily_returns'].dropna(), 
#               daily_prices[daily_prices['ticker'] == 'IRM']['daily_returns'].dropna())


# Growth (Daily Returns)


ProgrammingError: (mysql.connector.errors.ProgrammingError) 1054 (42S22): Unknown column 'nan' in 'field list'
[SQL: UPDATE daily_prices SET daily_return = %(dr)s WHERE ticker = %(t)s AND `date` = %(d)s]
[parameters: {'dr': nan, 't': 'DLR', 'd': '2025-06-09'}]
(Background on this error at: https://sqlalche.me/e/20/f405)